# Phase 1: Data Exploration

Understanding the shape, structure, and quality of the dataset before analyzing it.

In [ ]:
import os
import duckdb
import polars as pl
from dotenv import load_dotenv

load_dotenv(override=True)
dataset_path = os.getenv("LOCAL_DATASET_PATH")
con = duckdb.connect(database=":memory:")

# Load all columns as strings to avoid type-guessing on messy data
con.execute("""
    CREATE TEMP TABLE contracts AS
    SELECT * FROM read_csv_auto(
        '{0}', delim=',', header=true, strict_mode=false, all_varchar=true, parallel=false
    )
""".format(dataset_path))

## 1. How big is the dataset?

In [6]:
con.sql("SELECT COUNT(*) AS total_rows FROM contracts").pl()

total_rows
i64
1261329


In [ ]:
# Instrument type breakdown
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BLUE, RED, ORANGE, GREEN, GRAY = '#4878A8', '#D04040', '#E8A040', '#6BAF6B', '#888888'

inst = con.sql("""
    SELECT COALESCE(instrument_type, 'Missing') AS type, COUNT(*) AS cnt
    FROM contracts GROUP BY type ORDER BY cnt DESC
""").pl()

colors = {'C': BLUE, 'Missing': '#CCCCCC', 'A': ORANGE, 'SOSA': GREEN}
bar_colors = [colors.get(t, GRAY) for t in inst["type"].to_list()]

fig = go.Figure(go.Bar(
    x=inst["type"].to_list(), y=inst["cnt"].to_list(),
    marker_color=bar_colors,
    text=[f'{v/1000:.0f}K' for v in inst["cnt"].to_list()],
    textposition='outside',
    cliponaxis=False
))
fig.update_layout(
    title="What does each row represent?",
    yaxis_title="Row count",
    yaxis_tickformat=',',
    template="plotly_white",
    margin=dict(t=60),
    height=450
)
fig.show()

## Column overview

1.26M rows, 43 columns -- all loaded as VARCHAR to avoid type-guessing issues.

In [8]:
con.sql("DESCRIBE contracts").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""reference_number""","""VARCHAR""","""YES""",null,null,null
"""procurement_id""","""VARCHAR""","""YES""",null,null,null
"""vendor_name""","""VARCHAR""","""YES""",null,null,null
"""vendor_postal_code""","""VARCHAR""","""YES""",null,null,null
"""buyer_name""","""VARCHAR""","""YES""",null,null,null
…,…,…,…,…,…
"""award_criteria""","""VARCHAR""","""YES""",null,null,null
"""socioeconomic_indicator""","""VARCHAR""","""YES""",null,null,null
"""reporting_period""","""VARCHAR""","""YES""",null,null,null


## Row semantics

Each row is a transaction, not a unique contract. The `instrument_type` column tells us:
- **C** = new contract
- **A** = amendment to an existing contract
- **SOSA** = standing offer/supply arrangement (framework, not spending)

One contract can appear as multiple rows. `procurement_id` links them.

In [ ]:
# Contracts vs amendments vs standing offers
con.sql("""
    SELECT 
        instrument_type,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM contracts
    GROUP BY instrument_type
    ORDER BY row_count DESC
""").pl()

In [ ]:
# Unique contracts vs total rows
con.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT procurement_id) AS unique_procurement_ids,
        COUNT(*) - COUNT(DISTINCT procurement_id) AS duplicate_rows_from_amendments
    FROM contracts
""").pl()

## Missing data

Many fields only became mandatory after 2019 or 2022, so older nulls are by design. Counts both NULL and empty strings as missing.

In [ ]:
# Null/empty count and percentage for every column
total = con.sql("SELECT COUNT(*) FROM contracts").fetchone()[0]

rows = []
columns = [
    'reference_number', 'procurement_id', 'vendor_name', 'vendor_postal_code',
    'buyer_name', 'contract_date', 'economic_object_code', 'description_en',
    'description_fr', 'contract_period_start', 'delivery_date', 'contract_value',
    'original_value', 'amendment_value', 'comments_en', 'comments_fr',
    'additional_comments_en', 'additional_comments_fr', 'agreement_type_code',
    'trade_agreement', 'land_claims', 'commodity_type', 'commodity_code',
    'country_of_vendor', 'solicitation_procedure', 'limited_tendering_reason',
    'trade_agreement_exceptions', 'indigenous_business', 'indigenous_business_excluding_psib',
    'intellectual_property', 'potential_commercial_exploitation', 'former_public_servant',
    'contracting_entity', 'standing_offer_number', 'instrument_type', 'ministers_office',
    'number_of_bids', 'article_6_exceptions', 'award_criteria', 'socioeconomic_indicator',
    'reporting_period', 'owner_org', 'owner_org_title'
]

for col in columns:
    missing = con.sql(f"""
        SELECT COUNT(*) FROM contracts 
        WHERE {col} IS NULL OR TRIM({col}) = ''
    """).fetchone()[0]
    rows.append((col, missing, round(missing * 100.0 / total, 1)))

with pl.Config(tbl_rows=43):
    display(pl.DataFrame(rows, schema=["column_name", "missing_count", "missing_pct"], orient="row"))

In [ ]:
# Missing data - top 20 columns
missing_df = pl.DataFrame(rows, schema=["column_name", "missing_count", "missing_pct"], orient="row")
top_missing = missing_df.filter(pl.col("missing_pct") > 0).sort("missing_pct", descending=True).head(20)

cols = top_missing["column_name"].to_list()
pcts = top_missing["missing_pct"].to_list()
colors_m = [RED if p > 50 else ORANGE if p > 20 else BLUE for p in pcts]

fig = go.Figure(go.Bar(
    y=cols, x=pcts,
    orientation='h',
    marker_color=colors_m,
    text=[f'{p}%' for p in pcts],
    textposition='outside',
    cliponaxis=False
))
fig.update_layout(
    title="Missing data by column (red >50%, orange >20%)",
    xaxis_title="% missing",
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    margin=dict(t=60, l=250),
    height=600
)
fig.show()

## Time coverage

Canada's fiscal year runs April to March. `reporting_period` uses the format `YYYY-YYYY-QX`. Only mandatory after 2019-01-01.

In [13]:
# Earliest and latest reporting periods
con.sql("""
    SELECT 
        MIN(reporting_period) AS earliest,
        MAX(reporting_period) AS latest,
        COUNT(DISTINCT reporting_period) AS distinct_quarters
    FROM contracts
""").pl()

earliest,latest,distinct_quarters
str,str,i64
"""1""","""Q4""",142


In [14]:
# Row count by fiscal year (extract the first 9 characters: "YYYY-YYYY")
con.sql("""
    SELECT 
        LEFT(reporting_period, 9) AS fiscal_year,
        COUNT(*) AS row_count
    FROM contracts
    WHERE reporting_period IS NOT NULL
    GROUP BY fiscal_year
    ORDER BY fiscal_year
""").pl()

fiscal_year,row_count
str,i64
"""1""",6
"""10""",11
"""11""",15
"""12""",15
"""2""",11
…,…
"""C-2016-20""",1
"""Q1""",162
"""Q2""",2


In [15]:
# Row count by fiscal year - visual
fy = con.sql("""
    SELECT LEFT(reporting_period, 9) AS fiscal_year, COUNT(*) AS row_count
    FROM contracts
    WHERE reporting_period LIKE '____-____-Q_'
      AND LEFT(reporting_period, 9) >= '2004-2005' AND LEFT(reporting_period, 9) <= '2024-2025'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl()

fig = go.Figure(go.Bar(
    x=fy["fiscal_year"].to_list(), y=fy["row_count"].to_list(),
    marker_color=BLUE,
    text=[f'{v/1000:.0f}K' for v in fy["row_count"].to_list()],
    textposition='outside',
    cliponaxis=False
))
fig.update_layout(
    title="Data volume grew significantly after 2016-2017",
    yaxis_title="Rows",
    yaxis_tickformat=',',
    xaxis_tickangle=-45,
    template="plotly_white",
    margin=dict(t=60),
    height=500
)
fig.show()

## Financial fields

All stored as strings. Using `TRY_CAST` so bad values become NULL instead of crashing.

In [ ]:
# Cast failure check on financial fields
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(contract_value) - COUNT(TRY_CAST(contract_value AS DOUBLE)) AS contract_value_cast_failures,
        COUNT(original_value) - COUNT(TRY_CAST(original_value AS DOUBLE)) AS original_value_cast_failures,
        COUNT(amendment_value) - COUNT(TRY_CAST(amendment_value AS DOUBLE)) AS amendment_value_cast_failures
    FROM contracts
""").pl()

In [ ]:
# Financial field ranges
con.sql("""
    SELECT
        MIN(TRY_CAST(contract_value AS DOUBLE)) AS min_contract_value,
        MAX(TRY_CAST(contract_value AS DOUBLE)) AS max_contract_value,
        ROUND(AVG(TRY_CAST(contract_value AS DOUBLE)), 2) AS avg_contract_value,
        ROUND(MEDIAN(TRY_CAST(contract_value AS DOUBLE)), 2) AS median_contract_value,
        MIN(TRY_CAST(original_value AS DOUBLE)) AS min_original_value,
        MAX(TRY_CAST(original_value AS DOUBLE)) AS max_original_value,
        MIN(TRY_CAST(amendment_value AS DOUBLE)) AS min_amendment_value,
        MAX(TRY_CAST(amendment_value AS DOUBLE)) AS max_amendment_value
    FROM contracts
""").pl()

## Key categorical fields

Financial data is clean (zero cast failures). Now checking what kinds of contracts are in the data -- goods vs services, competitive vs sole-source.

In [ ]:
# Goods vs Services vs Construction
con.sql("""
    SELECT 
        commodity_type,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM contracts
    GROUP BY commodity_type
    ORDER BY row_count DESC
""").pl()

In [19]:
# Commodity type and solicitation procedure side by side
comm = con.sql("""
    SELECT CASE commodity_type WHEN 'S' THEN 'Services' WHEN 'G' THEN 'Goods' WHEN 'C' THEN 'Construction' ELSE 'Missing' END AS type,
        COUNT(*) AS cnt
    FROM contracts GROUP BY type ORDER BY cnt DESC
""").pl()

sol = con.sql("""
    SELECT COALESCE(solicitation_procedure, 'Missing') AS proc, COUNT(*) AS cnt
    FROM contracts GROUP BY proc ORDER BY cnt DESC
""").pl()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Commodity Type", "Solicitation Procedure"))

comm_colors = [BLUE, ORANGE, GREEN, '#CCCCCC']
fig.add_trace(go.Bar(
    x=comm["type"].to_list(), y=comm["cnt"].to_list(),
    marker_color=comm_colors[:len(comm)],
    text=[f'{v/1000:.0f}K' for v in comm["cnt"].to_list()],
    textposition='outside',
    cliponaxis=False,
    showlegend=False
), row=1, col=1)

sol_colors = ['#CCCCCC', BLUE, RED, GREEN, ORANGE, GRAY]
fig.add_trace(go.Bar(
    x=sol["proc"].to_list(), y=sol["cnt"].to_list(),
    marker_color=sol_colors[:len(sol)],
    text=[f'{v/1000:.0f}K' for v in sol["cnt"].to_list()],
    textposition='outside',
    cliponaxis=False,
    showlegend=False
), row=1, col=2)

fig.update_layout(
    template="plotly_white",
    margin=dict(t=60),
    height=500
)
fig.show()

In [ ]:
# Solicitation procedure breakdown
con.sql("""
    SELECT 
        solicitation_procedure,
        CASE solicitation_procedure
            WHEN 'OB' THEN 'Open Bidding (GETS)'
            WHEN 'TC' THEN 'Traditional Competitive'
            WHEN 'TN' THEN 'Non-Competitive'
            WHEN 'AC' THEN 'Advance Contract Award Notice'
            WHEN 'ST' THEN 'Selective Tendering'
            ELSE 'Unknown / NULL'
        END AS description,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM contracts
    GROUP BY solicitation_procedure
    ORDER BY row_count DESC
""").pl()

In [ ]:
# Top 15 vendor countries
con.sql("""
    SELECT 
        country_of_vendor,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM contracts
    WHERE country_of_vendor IS NOT NULL
    GROUP BY country_of_vendor
    ORDER BY row_count DESC
    LIMIT 15
""").pl()

## Top departments and vendors

In [ ]:
# Top 15 departments by row count
con.sql("""
    SELECT 
        SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        COUNT(*) AS row_count,
        COUNT(DISTINCT procurement_id) AS unique_contracts
    FROM contracts
    GROUP BY owner_org_title
    ORDER BY row_count DESC
    LIMIT 15
""").pl()

In [ ]:
# Top 15 vendors by contract count (original contracts only)
con.sql("""
    SELECT 
        vendor_name,
        COUNT(*) AS contract_count,
        ROUND(SUM(TRY_CAST(contract_value AS DOUBLE)), 2) AS total_value
    FROM contracts
    WHERE instrument_type = 'C'
      AND vendor_name IS NOT NULL
    GROUP BY vendor_name
    ORDER BY contract_count DESC
    LIMIT 15
""").pl()

In [ ]:
# Top 15 vendors by total dollar value (original contracts only)
con.sql("""
    SELECT 
        vendor_name,
        COUNT(*) AS contract_count,
        ROUND(SUM(TRY_CAST(contract_value AS DOUBLE)), 2) AS total_value
    FROM contracts
    WHERE instrument_type = 'C'
      AND vendor_name IS NOT NULL
    GROUP BY vendor_name
    ORDER BY total_value DESC
    LIMIT 15
""").pl()

## Sample rows

Checking actual records to spot formatting patterns, bilingual fields, and how amendments relate to their parent contract.

In [ ]:
# Example: a contract with its amendments linked by procurement_id
con.sql("""
    SELECT 
        procurement_id, instrument_type, vendor_name,
        contract_value, original_value, amendment_value,
        contract_date, reporting_period
    FROM contracts
    WHERE procurement_id IN (
        SELECT procurement_id 
        FROM contracts 
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id 
        HAVING COUNT(*) >= 3
        LIMIT 1
    )
    ORDER BY reporting_period
""").pl()

## Takeaways

- **1.26M rows, 43 columns**, spanning mid-2000s to 2025
- **Each row is a transaction**, not a unique contract -- must filter by `instrument_type`
- **28% of rows have no instrument_type** -- older records before the field was mandatory
- **Missing data follows a pattern** -- fields became mandatory in phases (2019, 2022), so older gaps are expected
- **Financial fields cast cleanly** -- zero failures, but values range from negative to $20B (heavily skewed)
- **`reporting_period` records when a contract was reported**, not awarded. Only mandatory after 2019-01-01. Pre-2019 data is ~22% missing.
- **National Defence dominates** -- 343K rows (27%), multi-billion shipbuilding/fighter jet contracts. Should be excluded from general analysis to avoid skewing averages.

### Investigation seeds for Phase 2

1. **Quarterly patterns** -- Q4 (Jan-Mar) is fiscal year-end. Do departments rush spending before budget resets?
2. **Amendments** -- 15% of rows are amendments. How much do contracts grow after award?
3. **Vendor landscape** -- Top vendors by count vs by value are very different lists. How concentrated is spending?